# Notebook 08 — Final Portfolio Selection, Risk Contribution and Institutional Recommendation

## Institutional Mandate

The Serendib University Endowment Fund is treated as a long-term institutional
investor with a conservative risk profile.

### Investment Horizon

The portfolio is intended as a long-term strategic allocation for a perpetual
university endowment, subject to periodic monitoring and review.

### Investment Objective

The objective is to preserve and grow capital over the long term while
supporting sustainable real growth and future university spending needs.

A precise numerical required-return target is not imposed because the brief
does not provide an explicit spending rate, inflation target or investment-cost
assumption.

Therefore, the final decision emphasizes capital preservation, downside-risk
control and portfolio resilience rather than maximum expected return.

### Portfolio Constraints

The candidate portfolios were constructed under the following modelling
constraints:

- prescribed seven-asset investable universe;
- long-only positions;
- fully invested portfolio;
- no leverage;
- between 2 and 7 included assets;
- 5% minimum weight for each included asset during candidate construction.

The 5% minimum position size is a modelling assumption used to avoid nominal
holdings and is not interpreted as a formal client policy.

### Decision Evidence

The final recommendation integrates:

1. out-of-sample performance;
2. 99% Historical VaR backtesting;
3. historical and hypothetical stress resilience;
4. capital concentration;
5. volatility risk contribution;
6. marginal and component VaR; and
7. Expected Shortfall contribution.

In [1]:
from pathlib import Path

import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    160
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.6f}"
)

In [2]:
PROJECT_ROOT = Path("..")

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)

FIGURE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Project root:",
    PROJECT_ROOT.resolve()
)

Project root: C:\quant_edge_round2


In [3]:
simple_returns = pd.read_csv(
    PROCESSED_DIR
    / "simple_returns.csv",
    index_col=0,
    parse_dates=True
)

simple_returns = (
    simple_returns
    .sort_index()
)

print(
    "Return data shape:",
    simple_returns.shape
)

print(
    "Start:",
    simple_returns.index.min()
)

print(
    "End:",
    simple_returns.index.max()
)

simple_returns.head()

Return data shape: (5512, 7)
Start: 2004-11-19 00:00:00
End: 2026-10-19 00:00:00


,SPY,EFA,EEM,AGG,GLD,XLE,XLK
2004-11-19,-0.011117,-0.006474,-0.014297,-0.002044,0.009013,0.011961,-0.015464
2004-11-22,0.004769,0.002933,0.003903,0.001561,0.003796,0.015943,0.001428
2004-11-23,0.001526,0.001949,0.001733,-0.000585,-0.004449,0.006223,-0.004278
2004-11-24,0.002370,0.004539,0.010647,0.000974,0.006704,0.008335,0.008115
2004-11-26,-0.000760,0.005811,0.016606,-0.002920,0.005327,0.004267,-0.001420


In [4]:
TRAIN_END = "2021-12-31"

training_returns = (
    simple_returns
    .loc[
        simple_returns.index
        <= TRAIN_END
    ]
)

print(
    "Training observations:",
    len(training_returns)
)

print(
    "Training end:",
    training_returns.index.max()
)

Training observations: 4309
Training end: 2021-12-31 00:00:00


In [5]:
shortlist = pd.read_csv(
    TABLE_DIR
    / "portfolio_shortlist.csv"
)

shortlist

,Portfolio Label,N,Subset,Method,Weights
0,Equal Weight Benchmark,7,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",Equal Weight,"[0.1428571429, 0.1428571429, 0.1428571429, 0.1..."
1,Best GMVP,2,"SPY, AGG",GMVP,"[0.0787430197, 0.9212569803]"
2,Best Maximum Sharpe,3,"AGG, GLD, XLK",Maximum Sharpe,"[0.6522812131, 0.1292391967, 0.2184795901]"
3,Best Risk Parity,3,"AGG, GLD, XLK",Risk Parity,"[0.6545410719, 0.1779072155, 0.1675517126]"
4,Best Minimum ES,2,"SPY, AGG",Minimum ES,"[0.0733156417, 0.9266843583]"


In [6]:
def parse_weights(value):

    if isinstance(
        value,
        str
    ):
        value = ast.literal_eval(
            value
        )

    return np.asarray(
        value,
        dtype=float
    )

In [7]:
weight_validation = []

for _, row in shortlist.iterrows():

    assets = [
        x.strip()
        for x in row[
            "Subset"
        ].split(",")
    ]

    weights = parse_weights(
        row["Weights"]
    )

    weight_validation.append(
        {
            "Portfolio":
                row[
                    "Portfolio Label"
                ],

            "Number of Assets":
                len(assets),

            "Weight Sum":
                weights.sum(),

            "Minimum Weight":
                weights.min(),

            "Maximum Weight":
                weights.max()
        }
    )

weight_validation = (
    pd.DataFrame(
        weight_validation
    )
    .set_index(
        "Portfolio"
    )
)

weight_validation

,Number of Assets,Weight Sum,Minimum Weight,Maximum Weight
Portfolio,,,,
Equal Weight Benchmark,7,1.000000,0.142857,0.142857
Best GMVP,2,1.000000,0.078743,0.921257
Best Maximum Sharpe,3,1.000000,0.129239,0.652281
Best Risk Parity,3,1.000000,0.167552,0.654541
Best Minimum ES,2,1.000000,0.073316,0.926684


In [8]:
assert np.allclose(
    weight_validation[
        "Weight Sum"
    ],
    1.0
)

assert (
    weight_validation[
        "Minimum Weight"
    ]
    >= -1e-10
).all()

print(
    "Portfolio weights validated."
)

Portfolio weights validated.


In [9]:
TRADING_DAYS = 252

annual_covariance = (
    training_returns.cov()
    * TRADING_DAYS
)

annual_covariance.round(6)

,SPY,EFA,EEM,AGG,GLD,XLE,XLK
SPY,0.037009,0.037868,0.047555,-0.000649,0.001477,0.045617,0.037790
EFA,0.037868,0.048038,0.056491,-0.000343,0.005839,0.050561,0.037783
EEM,0.047555,0.056491,0.085732,-0.000920,0.008685,0.064953,0.048734
AGG,-0.000649,-0.000343,-0.000920,0.002373,0.001512,-0.000739,-0.000818
GLD,0.001477,0.005839,0.008685,0.001512,0.032384,0.008715,0.001088
XLE,0.045617,0.050561,0.064953,-0.000739,0.008715,0.094253,0.041709
XLK,0.037790,0.037783,0.048734,-0.000818,0.001088,0.041709,0.046020


In [10]:
def calculate_risk_contribution(
    assets,
    weights,
    covariance_matrix
):

    covariance = (
        covariance_matrix
        .loc[
            assets,
            assets
        ]
        .values
    )

    portfolio_variance = (
        weights.T
        @ covariance
        @ weights
    )

    portfolio_volatility = (
        np.sqrt(
            portfolio_variance
        )
    )

    marginal_contribution = (
        covariance
        @ weights
    ) / portfolio_volatility

    component_contribution = (
        weights
        * marginal_contribution
    )

    percentage_contribution = (
        component_contribution
        / portfolio_volatility
    )

    result = pd.DataFrame(
        {
            "Asset":
                assets,

            "Weight":
                weights,

            "Marginal Risk Contribution":
                marginal_contribution,

            "Component Risk Contribution":
                component_contribution,

            "Risk Contribution":
                percentage_contribution
        }
    )

    return (
        portfolio_volatility,
        result
    )

In [11]:
risk_contribution_rows = []

portfolio_risk_summary = []

for _, row in shortlist.iterrows():

    portfolio = (
        row[
            "Portfolio Label"
        ]
    )

    assets = [
        x.strip()
        for x in row[
            "Subset"
        ].split(",")
    ]

    weights = parse_weights(
        row["Weights"]
    )

    (
        portfolio_volatility,
        contribution_table
    ) = calculate_risk_contribution(
        assets,
        weights,
        annual_covariance
    )

    contribution_table[
        "Portfolio"
    ] = portfolio

    risk_contribution_rows.append(
        contribution_table
    )

    portfolio_risk_summary.append(
        {
            "Portfolio":
                portfolio,

            "Training Volatility":
                portfolio_volatility,

            "Largest Risk Contribution":
                contribution_table[
                    "Risk Contribution"
                ].max(),

            "Largest Risk Contributor":
                contribution_table.loc[
                    contribution_table[
                        "Risk Contribution"
                    ].idxmax(),
                    "Asset"
                ]
        }
    )

risk_contributions = pd.concat(
    risk_contribution_rows,
    ignore_index=True
)

portfolio_risk_summary = (
    pd.DataFrame(
        portfolio_risk_summary
    )
    .set_index(
        "Portfolio"
    )
)

portfolio_risk_summary

,Training Volatility,Largest Risk Contribution,Largest Risk Contributor
Portfolio,,,
Equal Weight Benchmark,0.164852,0.233721,EEM
Best GMVP,0.046357,0.915120,AGG
Best Maximum Sharpe,0.061888,0.551101,XLK
Best Risk Parity,0.059757,0.345754,XLK
Best Minimum ES,0.046349,0.927918,AGG


In [12]:
risk_contribution_display = (
    risk_contributions.copy()
)

risk_contribution_display[
    "Weight"
] *= 100

risk_contribution_display[
    "Risk Contribution"
] *= 100

risk_contribution_display[
    [
        "Portfolio",
        "Asset",
        "Weight",
        "Risk Contribution"
    ]
].round(2)

,Portfolio,Asset,Weight,Risk Contribution
0,Equal Weight Benchmark,SPY,14.290000,15.520000
1,Equal Weight Benchmark,EFA,14.290000,17.740000
2,Equal Weight Benchmark,EEM,14.290000,23.370000
3,Equal Weight Benchmark,AGG,14.290000,0.030000
4,Equal Weight Benchmark,GLD,14.290000,4.480000
5,Equal Weight Benchmark,XLE,14.290000,22.910000
6,Equal Weight Benchmark,XLK,14.290000,15.940000
7,Best GMVP,SPY,7.870000,8.490000
8,Best GMVP,AGG,92.130000,91.510000
9,Best Maximum Sharpe,AGG,65.230000,26.640000


In [13]:
risk_sum_check = (
    risk_contributions
    .groupby(
        "Portfolio"
    )[
        "Risk Contribution"
    ]
    .sum()
)

risk_sum_check

Portfolio
Best GMVP                1.000000
Best Maximum Sharpe      1.000000
Best Minimum ES          1.000000
Best Risk Parity         1.000000
Equal Weight Benchmark   1.000000
Name: Risk Contribution, dtype: float64

In [14]:
assert np.allclose(
    risk_sum_check.values,
    1.0
)

print(
    "Risk contributions sum to 100%."
)

Risk contributions sum to 100%.


In [15]:
risk_contribution_matrix = (
    risk_contributions
    .pivot(
        index="Portfolio",
        columns="Asset",
        values="Risk Contribution"
    )
    .fillna(0)
)

(
    risk_contribution_matrix
    * 100
).round(2)

Asset,AGG,EEM,EFA,GLD,SPY,XLE,XLK
Portfolio,,,,,,,
Best GMVP,91.510000,0.000000,0.000000,0.000000,8.490000,0.000000,0.000000
Best Maximum Sharpe,26.640000,0.000000,0.000000,18.250000,0.000000,0.000000,55.110000
Best Minimum ES,92.790000,0.000000,0.000000,0.000000,7.210000,0.000000,0.000000
Best Risk Parity,30.880000,0.000000,0.000000,34.540000,0.000000,0.000000,34.580000
Equal Weight Benchmark,0.030000,23.370000,17.740000,4.480000,15.520000,22.910000,15.940000


In [16]:
(
    risk_contribution_matrix
    * 100
).plot(
    kind="bar",
    stacked=True,
    figsize=(12, 7)
)

plt.ylabel(
    "Contribution to Portfolio Volatility (%)"
)

plt.title(
    "Portfolio Risk Contributions"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.legend(
    title="Asset",
    bbox_to_anchor=(
        1.02,
        1
    )
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "portfolio_risk_contributions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_34172\3785486365.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
diversification_rows = []

for _, row in shortlist.iterrows():

    portfolio = (
        row[
            "Portfolio Label"
        ]
    )

    assets = [
        x.strip()
        for x in row[
            "Subset"
        ].split(",")
    ]

    weights = parse_weights(
        row["Weights"]
    )

    hhi = np.sum(
        weights ** 2
    )

    effective_n = (
        1 / hhi
    )

    diversification_rows.append(
        {
            "Portfolio":
                portfolio,

            "Number of Assets":
                len(assets),

            "Largest Weight":
                weights.max(),

            "HHI":
                hhi,

            "Effective Number of Holdings":
                effective_n
        }
    )

diversification_summary = (
    pd.DataFrame(
        diversification_rows
    )
    .set_index(
        "Portfolio"
    )
)

diversification_summary

,Number of Assets,Largest Weight,HHI,Effective Number of Holdings
Portfolio,,,,
Equal Weight Benchmark,7,0.142857,0.142857,7.000000
Best GMVP,2,0.921257,0.854915,1.169707
Best Maximum Sharpe,3,0.652281,0.489907,2.041204
Best Risk Parity,3,0.654541,0.488149,2.048557
Best Minimum ES,2,0.926684,0.864119,1.157248


In [18]:
diversification_display = (
    diversification_summary
    .copy()
)

diversification_display[
    "Largest Weight"
] *= 100

diversification_display.round(2)

,Number of Assets,Largest Weight,HHI,Effective Number of Holdings
Portfolio,,,,
Equal Weight Benchmark,7,14.290000,0.140000,7.000000
Best GMVP,2,92.130000,0.850000,1.170000
Best Maximum Sharpe,3,65.230000,0.490000,2.040000
Best Risk Parity,3,65.450000,0.490000,2.050000
Best Minimum ES,2,92.670000,0.860000,1.160000


In [19]:
[
    x.name
    for x in TABLE_DIR.glob(
        "*.csv"
    )
    if "performance" in x.name.lower()
    or "oos" in x.name.lower()
]

['institutional_performance_summary_development.csv',
 'oos_performance_rankings.csv',
 'oos_performance_summary.csv',
 'oos_portfolio_returns.csv']

In [20]:
oos_performance = pd.read_csv(
    TABLE_DIR
    / "oos_performance_summary.csv",
    index_col=0
)

oos_performance

,Annualized Return,CAGR,Volatility,Sharpe Ratio,Sortino Ratio,Maximum Drawdown,Calmar Ratio,Terminal Wealth
Portfolio,,,,,,,,
Equal Weight Benchmark,0.137599,0.137391,0.132982,0.731109,1.050418,-0.173676,0.791077,1.848855
Best GMVP,0.007878,0.005984,0.061873,-0.525208,-0.721078,-0.164387,0.036399,1.028889
Best Maximum Sharpe,0.066331,0.064616,0.086208,0.301092,0.434318,-0.181008,0.356976,1.348383
Best Risk Parity,0.065134,0.063723,0.081931,0.302203,0.433203,-0.171095,0.372442,1.342994
Best Minimum ES,0.007167,0.005273,0.061801,-0.537322,-0.737413,-0.164101,0.032133,1.025424


In [21]:
oos_performance.columns.tolist()

['Annualized Return',
 'CAGR',
 'Volatility',
 'Sharpe Ratio',
 'Sortino Ratio',
 'Maximum Drawdown',
 'Calmar Ratio',
 'Terminal Wealth']

In [22]:
oos_decision_metrics = (
    oos_performance[
        [
            "Annualized Return",
            "CAGR",
            "Volatility",
            "Sharpe Ratio",
            "Sortino Ratio",
            "Maximum Drawdown",
            "Calmar Ratio",
            "Terminal Wealth"
        ]
    ]
    .copy()
)

print(
    "OOS decision metrics loaded:",
    oos_decision_metrics.shape
)

oos_decision_metrics

OOS decision metrics loaded: (5, 8)


,Annualized Return,CAGR,Volatility,Sharpe Ratio,Sortino Ratio,Maximum Drawdown,Calmar Ratio,Terminal Wealth
Portfolio,,,,,,,,
Equal Weight Benchmark,0.137599,0.137391,0.132982,0.731109,1.050418,-0.173676,0.791077,1.848855
Best GMVP,0.007878,0.005984,0.061873,-0.525208,-0.721078,-0.164387,0.036399,1.028889
Best Maximum Sharpe,0.066331,0.064616,0.086208,0.301092,0.434318,-0.181008,0.356976,1.348383
Best Risk Parity,0.065134,0.063723,0.081931,0.302203,0.433203,-0.171095,0.372442,1.342994
Best Minimum ES,0.007167,0.005273,0.061801,-0.537322,-0.737413,-0.164101,0.032133,1.025424


In [23]:
oos_display = oos_decision_metrics.copy()

percentage_columns = [
    "Annualized Return",
    "CAGR",
    "Volatility",
    "Maximum Drawdown"
]

oos_display[
    percentage_columns
] *= 100

oos_display.round(2)

,Annualized Return,CAGR,Volatility,Sharpe Ratio,Sortino Ratio,Maximum Drawdown,Calmar Ratio,Terminal Wealth
Portfolio,,,,,,,,
Equal Weight Benchmark,13.760000,13.740000,13.300000,0.730000,1.050000,-17.370000,0.790000,1.850000
Best GMVP,0.790000,0.600000,6.190000,-0.530000,-0.720000,-16.440000,0.040000,1.030000
Best Maximum Sharpe,6.630000,6.460000,8.620000,0.300000,0.430000,-18.100000,0.360000,1.350000
Best Risk Parity,6.510000,6.370000,8.190000,0.300000,0.430000,-17.110000,0.370000,1.340000
Best Minimum ES,0.720000,0.530000,6.180000,-0.540000,-0.740000,-16.410000,0.030000,1.030000


In [24]:
[
    x.name
    for x in TABLE_DIR.glob("*.csv")
    if (
        "validation" in x.name.lower()
        or "kupiec" in x.name.lower()
        or "christoffersen" in x.name.lower()
        or "coverage" in x.name.lower()
    )
]

['christoffersen_independence_results.csv',
 'conditional_coverage_results.csv',
 'kupiec_test_results.csv',
 'var_validation_99_summary.csv',
 'var_validation_scorecard.csv']

In [25]:
var_validation_99 = pd.read_csv(
    TABLE_DIR
    / "var_validation_99_summary.csv",
    index_col=0
)

var_validation_99

,Model,Exceedances,Observed Rate,Kupiec p-value,Kupiec Pass,Christoffersen p-value,Independence Pass,Conditional Coverage p-value,Conditional Coverage Pass,Latest 250D Exceptions,Latest Basel Zone,Overall Validation Pass
Portfolio,,,,,,,,,,,,
Best GMVP,Historical,15,0.012469,0.407251,True,0.538076,True,0.586862,True,2,Green,True
Best GMVP,Parametric Normal,26,0.021613,0.000453,False,0.589453,True,0.001844,False,4,Green,False
Best Maximum Sharpe,Historical,15,0.012469,0.407251,True,0.177659,True,0.285944,True,4,Green,True
Best Maximum Sharpe,Parametric Normal,20,0.016625,0.034971,False,0.042063,False,0.013710,False,5,Yellow,False
Best Minimum ES,Historical,15,0.012469,0.407251,True,0.538076,True,0.586862,True,2,Green,True
Best Minimum ES,Parametric Normal,27,0.022444,0.000192,False,0.635224,True,0.000855,False,3,Green,False
Best Risk Parity,Historical,15,0.012469,0.407251,True,0.177659,True,0.285944,True,4,Green,True
Best Risk Parity,Parametric Normal,26,0.021613,0.000453,False,0.589453,True,0.001844,False,11,Red,False
Equal Weight Benchmark,Historical,17,0.014131,0.175150,True,0.236405,True,0.197878,True,3,Green,True


In [26]:
var_validation_99.columns.tolist()

['Model',
 'Exceedances',
 'Observed Rate',
 'Kupiec p-value',
 'Kupiec Pass',
 'Christoffersen p-value',
 'Independence Pass',
 'Conditional Coverage p-value',
 'Conditional Coverage Pass',
 'Latest 250D Exceptions',
 'Latest Basel Zone',
 'Overall Validation Pass']

In [27]:
var_validation_99["Model"].value_counts()

Model
Historical           5
Parametric Normal    5
Name: count, dtype: int64

In [28]:
historical_var_validation = (
    var_validation_99[
        var_validation_99["Model"]
        .str.contains(
            "Historical",
            case=False,
            na=False
        )
    ]
    .copy()
)

historical_var_validation

,Model,Exceedances,Observed Rate,Kupiec p-value,Kupiec Pass,Christoffersen p-value,Independence Pass,Conditional Coverage p-value,Conditional Coverage Pass,Latest 250D Exceptions,Latest Basel Zone,Overall Validation Pass
Portfolio,,,,,,,,,,,,
Best GMVP,Historical,15,0.012469,0.407251,True,0.538076,True,0.586862,True,2,Green,True
Best Maximum Sharpe,Historical,15,0.012469,0.407251,True,0.177659,True,0.285944,True,4,Green,True
Best Minimum ES,Historical,15,0.012469,0.407251,True,0.538076,True,0.586862,True,2,Green,True
Best Risk Parity,Historical,15,0.012469,0.407251,True,0.177659,True,0.285944,True,4,Green,True
Equal Weight Benchmark,Historical,17,0.014131,0.175150,True,0.236405,True,0.197878,True,3,Green,True


In [29]:
historical_var_decision = (
    historical_var_validation[
        [
            "Exceedances",
            "Observed Rate",
            "Kupiec Pass",
            "Independence Pass",
            "Conditional Coverage Pass",
            "Latest 250D Exceptions",
            "Latest Basel Zone",
            "Overall Validation Pass"
        ]
    ]
    .copy()
)

historical_var_decision

,Exceedances,Observed Rate,Kupiec Pass,Independence Pass,Conditional Coverage Pass,Latest 250D Exceptions,Latest Basel Zone,Overall Validation Pass
Portfolio,,,,,,,,
Best GMVP,15,0.012469,True,True,True,2,Green,True
Best Maximum Sharpe,15,0.012469,True,True,True,4,Green,True
Best Minimum ES,15,0.012469,True,True,True,2,Green,True
Best Risk Parity,15,0.012469,True,True,True,4,Green,True
Equal Weight Benchmark,17,0.014131,True,True,True,3,Green,True


In [30]:
historical_var_decision[
    [
        "Kupiec Pass",
        "Independence Pass",
        "Conditional Coverage Pass",
        "Overall Validation Pass"
    ]
]

,Kupiec Pass,Independence Pass,Conditional Coverage Pass,Overall Validation Pass
Portfolio,,,,
Best GMVP,True,True,True,True
Best Maximum Sharpe,True,True,True,True
Best Minimum ES,True,True,True,True
Best Risk Parity,True,True,True,True
Equal Weight Benchmark,True,True,True,True


In [31]:
assert (
    historical_var_decision[
        "Overall Validation Pass"
    ]
    .all()
)

print(
    "All shortlisted portfolios pass "
    "99% Historical VaR validation."
)

All shortlisted portfolios pass 99% Historical VaR validation.


In [32]:
stress_rankings = pd.read_csv(
    TABLE_DIR
    / "stress_resilience_rankings.csv",
    index_col=0
)

stress_rankings


,Historical Return Rank,Historical Drawdown Rank,Historical ES Rank,Hypothetical Worst-Shock Rank,Average Stress Rank,Overall Stress Rank
Portfolio,,,,,,
Best GMVP,3.000000,2.000000,2.000000,4.000000,2.750000,3.000000
Best Maximum Sharpe,4.000000,4.000000,4.000000,2.000000,3.500000,4.000000
Best Minimum ES,2.000000,1.000000,1.000000,3.000000,1.750000,1.000000
Best Risk Parity,1.000000,3.000000,3.000000,1.000000,2.000000,2.000000
Equal Weight Benchmark,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


In [33]:
stress_decision = pd.read_csv(
    TABLE_DIR
    / "final_stress_decision_table.csv",
    index_col=0
)

stress_decision

,Overall Stress Rank,Number of Assets,Largest Holding,Largest Weight
Portfolio,,,,
Best GMVP,3.000000,2,AGG,92.125698
Best Maximum Sharpe,4.000000,3,AGG,65.228121
Best Minimum ES,1.000000,2,AGG,92.668436
Best Risk Parity,2.000000,3,AGG,65.454107
Equal Weight Benchmark,5.000000,7,SPY,14.285714


In [34]:
stress_decision[
    "Overall Stress Rank"
].sort_values()

Portfolio
Best Minimum ES          1.000000
Best Risk Parity         2.000000
Best GMVP                3.000000
Best Maximum Sharpe      4.000000
Equal Weight Benchmark   5.000000
Name: Overall Stress Rank, dtype: float64

In [35]:
institutional_comparison = (
    oos_decision_metrics[
        [
            "Annualized Return",
            "Volatility",
            "Sharpe Ratio",
            "Sortino Ratio",
            "Maximum Drawdown",
            "Calmar Ratio"
        ]
    ]
    .join(
        diversification_summary[
            [
                "Largest Weight",
                "HHI",
                "Effective Number of Holdings"
            ]
        ]
    )
    .join(
        portfolio_risk_summary[
            [
                "Largest Risk Contributor",
                "Largest Risk Contribution"
            ]
        ]
    )
    .join(
        stress_decision[
            [
                "Overall Stress Rank"
            ]
        ]
    )
    .join(
        historical_var_decision[
            [
                "Observed Rate",
                "Overall Validation Pass",
                "Latest 250D Exceptions",
                "Latest Basel Zone"
            ]
        ]
    )
)

institutional_comparison

,Annualized Return,Volatility,Sharpe Ratio,Sortino Ratio,Maximum Drawdown,Calmar Ratio,Largest Weight,HHI,Effective Number of Holdings,Largest Risk Contributor,Largest Risk Contribution,Overall Stress Rank,Observed Rate,Overall Validation Pass,Latest 250D Exceptions,Latest Basel Zone
Portfolio,,,,,,,,,,,,,,,,
Equal Weight Benchmark,0.137599,0.132982,0.731109,1.050418,-0.173676,0.791077,0.142857,0.142857,7.000000,EEM,0.233721,5.000000,0.014131,True,3,Green
Best GMVP,0.007878,0.061873,-0.525208,-0.721078,-0.164387,0.036399,0.921257,0.854915,1.169707,AGG,0.915120,3.000000,0.012469,True,2,Green
Best Maximum Sharpe,0.066331,0.086208,0.301092,0.434318,-0.181008,0.356976,0.652281,0.489907,2.041204,XLK,0.551101,4.000000,0.012469,True,4,Green
Best Risk Parity,0.065134,0.081931,0.302203,0.433203,-0.171095,0.372442,0.654541,0.488149,2.048557,XLK,0.345754,2.000000,0.012469,True,4,Green
Best Minimum ES,0.007167,0.061801,-0.537322,-0.737413,-0.164101,0.032133,0.926684,0.864119,1.157248,AGG,0.927918,1.000000,0.012469,True,2,Green


In [36]:
institutional_display = (
    institutional_comparison.copy()
)

percentage_columns = [
    "Annualized Return",
    "Volatility",
    "Maximum Drawdown",
    "Largest Weight",
    "Largest Risk Contribution",
    "Observed Rate"
]

institutional_display[
    percentage_columns
] *= 100

institutional_display.round(2)

,Annualized Return,Volatility,Sharpe Ratio,Sortino Ratio,Maximum Drawdown,Calmar Ratio,Largest Weight,HHI,Effective Number of Holdings,Largest Risk Contributor,Largest Risk Contribution,Overall Stress Rank,Observed Rate,Overall Validation Pass,Latest 250D Exceptions,Latest Basel Zone
Portfolio,,,,,,,,,,,,,,,,
Equal Weight Benchmark,13.760000,13.300000,0.730000,1.050000,-17.370000,0.790000,14.290000,0.140000,7.000000,EEM,23.370000,5.000000,1.410000,True,3,Green
Best GMVP,0.790000,6.190000,-0.530000,-0.720000,-16.440000,0.040000,92.130000,0.850000,1.170000,AGG,91.510000,3.000000,1.250000,True,2,Green
Best Maximum Sharpe,6.630000,8.620000,0.300000,0.430000,-18.100000,0.360000,65.230000,0.490000,2.040000,XLK,55.110000,4.000000,1.250000,True,4,Green
Best Risk Parity,6.510000,8.190000,0.300000,0.430000,-17.110000,0.370000,65.450000,0.490000,2.050000,XLK,34.580000,2.000000,1.250000,True,4,Green
Best Minimum ES,0.720000,6.180000,-0.540000,-0.740000,-16.410000,0.030000,92.670000,0.860000,1.160000,AGG,92.790000,1.000000,1.250000,True,2,Green


In [37]:
print(
    institutional_comparison.index.tolist()
)

['Equal Weight Benchmark', 'Best GMVP', 'Best Maximum Sharpe', 'Best Risk Parity', 'Best Minimum ES']


In [38]:
assert (
    institutional_comparison.shape[0]
    == 5
)

assert (
    institutional_comparison[
        "Overall Validation Pass"
    ]
    .all()
)

print(
    "Institutional comparison table validated."
)

Institutional comparison table validated.


In [39]:
decision_gates = pd.DataFrame(
    index=institutional_comparison.index
)

decision_gates[
    "VaR Validation Gate"
] = (
    institutional_comparison[
        "Overall Validation Pass"
    ]
)

decision_gates[
    "Positive OOS Sharpe Gate"
] = (
    institutional_comparison[
        "Sharpe Ratio"
    ]
    > 0
)

decision_gates[
    "Stress Top-3 Gate"
] = (
    institutional_comparison[
        "Overall Stress Rank"
    ]
    <= 3
)

decision_gates[
    "Concentration Below 80% Gate"
] = (
    institutional_comparison[
        "Largest Weight"
    ]
    < 0.80
)

decision_gates

,VaR Validation Gate,Positive OOS Sharpe Gate,Stress Top-3 Gate,Concentration Below 80% Gate
Portfolio,,,,
Equal Weight Benchmark,True,True,False,True
Best GMVP,True,False,True,False
Best Maximum Sharpe,True,True,False,True
Best Risk Parity,True,True,True,True
Best Minimum ES,True,False,True,False


In [40]:
decision_gates[
    "Number of Gates Passed"
] = (
    decision_gates.sum(
        axis=1
    )
)

decision_gates.sort_values(
    "Number of Gates Passed",
    ascending=False
)

,VaR Validation Gate,Positive OOS Sharpe Gate,Stress Top-3 Gate,Concentration Below 80% Gate,Number of Gates Passed
Portfolio,,,,,
Best Risk Parity,True,True,True,True,4
Equal Weight Benchmark,True,True,False,True,3
Best Maximum Sharpe,True,True,False,True,3
Best GMVP,True,False,True,False,2
Best Minimum ES,True,False,True,False,2


### Decision-Gate Interpretation

The final decision gates are internal screening rules used to organize the
evidence consistently.

The positive Sharpe, top-three stress rank and 80% maximum-weight thresholds
are analytical decision heuristics rather than formal constraints supplied by
the client.

They are used to identify portfolios that simultaneously demonstrate credible
risk-adjusted performance, stress resilience and acceptable diversification.

In [41]:
comparison_metrics = (
    institutional_comparison[
        [
            "Annualized Return",
            "Volatility",
            "Maximum Drawdown"
        ]
    ]
    * 100
)

comparison_metrics.plot(
    kind="bar",
    figsize=(12, 7)
)

plt.axhline(
    0,
    linewidth=1
)

plt.ylabel(
    "Percent (%)"
)

plt.title(
    "Out-of-Sample Return and Risk Comparison"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "final_return_risk_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_34172\190724468.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [42]:
fig, ax = plt.subplots(
    figsize=(9, 6)
)

for portfolio in institutional_comparison.index:

    x = (
        institutional_comparison.loc[
            portfolio,
            "Effective Number of Holdings"
        ]
    )

    y = (
        institutional_comparison.loc[
            portfolio,
            "Overall Stress Rank"
        ]
    )

    ax.scatter(
        x,
        y,
        s=80
    )

    ax.annotate(
        portfolio,
        (
            x,
            y
        ),
        xytext=(
            5,
            5
        ),
        textcoords="offset points"
    )

ax.invert_yaxis()

ax.set_xlabel(
    "Effective Number of Holdings"
)

ax.set_ylabel(
    "Stress Rank (Lower is Better)"
)

ax.set_title(
    "Diversification versus Stress Resilience"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "diversification_vs_stress.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_34172\486417454.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [43]:
fig, ax = plt.subplots(
    figsize=(9, 6)
)

for portfolio in institutional_comparison.index:

    x = (
        institutional_comparison.loc[
            portfolio,
            "Sharpe Ratio"
        ]
    )

    y = (
        institutional_comparison.loc[
            portfolio,
            "Overall Stress Rank"
        ]
    )

    ax.scatter(
        x,
        y,
        s=80
    )

    ax.annotate(
        portfolio,
        (
            x,
            y
        ),
        xytext=(
            5,
            5
        ),
        textcoords="offset points"
    )

ax.invert_yaxis()

ax.axvline(
    0,
    linewidth=1
)

ax.set_xlabel(
    "Out-of-Sample Sharpe Ratio"
)

ax.set_ylabel(
    "Stress Rank (Lower is Better)"
)

ax.set_title(
    "Risk-Adjusted Performance versus Stress Resilience"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "sharpe_vs_stress.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_34172\2838310514.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [44]:
risk_parity_row = (
    shortlist[
        shortlist[
            "Portfolio Label"
        ]
        == "Best Risk Parity"
    ]
    .iloc[0]
)

risk_parity_assets = [
    x.strip()
    for x in risk_parity_row[
        "Subset"
    ].split(",")
]

risk_parity_weights = parse_weights(
    risk_parity_row[
        "Weights"
    ]
)

recommended_allocation = pd.DataFrame(
    {
        "Asset":
            risk_parity_assets,

        "Weight":
            risk_parity_weights
    }
)

recommended_allocation[
    "Weight (%)"
] = (
    recommended_allocation[
        "Weight"
    ]
    * 100
)

recommended_allocation[
    [
        "Asset",
        "Weight (%)"
    ]
].round(2)

,Asset,Weight (%)
0,AGG,65.450000
1,GLD,17.790000
2,XLK,16.760000


In [45]:
recommended_risk_contributions = (
    risk_contributions[
        risk_contributions[
            "Portfolio"
        ]
        == "Best Risk Parity"
    ]
    .copy()
)

recommended_risk_contributions[
    "Weight (%)"
] = (
    recommended_risk_contributions[
        "Weight"
    ]
    * 100
)

recommended_risk_contributions[
    "Risk Contribution (%)"
] = (
    recommended_risk_contributions[
        "Risk Contribution"
    ]
    * 100
)

recommended_risk_contributions[
    [
        "Asset",
        "Weight (%)",
        "Risk Contribution (%)"
    ]
].round(2)

,Asset,Weight (%),Risk Contribution (%)
12,AGG,65.450000,30.880000
13,GLD,17.790000,34.540000
14,XLK,16.760000,34.580000


In [46]:
final_evidence_table = (
    institutional_comparison[
        [
            "Annualized Return",
            "Volatility",
            "Sharpe Ratio",
            "Maximum Drawdown",
            "Largest Weight",
            "Effective Number of Holdings",
            "Overall Stress Rank",
            "Overall Validation Pass",
            "Latest Basel Zone"
        ]
    ]
    .copy()
)

final_evidence_table

,Annualized Return,Volatility,Sharpe Ratio,Maximum Drawdown,Largest Weight,Effective Number of Holdings,Overall Stress Rank,Overall Validation Pass,Latest Basel Zone
Portfolio,,,,,,,,,
Equal Weight Benchmark,0.137599,0.132982,0.731109,-0.173676,0.142857,7.000000,5.000000,True,Green
Best GMVP,0.007878,0.061873,-0.525208,-0.164387,0.921257,1.169707,3.000000,True,Green
Best Maximum Sharpe,0.066331,0.086208,0.301092,-0.181008,0.652281,2.041204,4.000000,True,Green
Best Risk Parity,0.065134,0.081931,0.302203,-0.171095,0.654541,2.048557,2.000000,True,Green
Best Minimum ES,0.007167,0.061801,-0.537322,-0.164101,0.926684,1.157248,1.000000,True,Green


In [47]:
final_evidence_display = (
    final_evidence_table.copy()
)

for col in [
    "Annualized Return",
    "Volatility",
    "Maximum Drawdown",
    "Largest Weight"
]:
    final_evidence_display[col] *= 100

final_evidence_display.round(2)

,Annualized Return,Volatility,Sharpe Ratio,Maximum Drawdown,Largest Weight,Effective Number of Holdings,Overall Stress Rank,Overall Validation Pass,Latest Basel Zone
Portfolio,,,,,,,,,
Equal Weight Benchmark,13.760000,13.300000,0.730000,-17.370000,14.290000,7.000000,5.000000,True,Green
Best GMVP,0.790000,6.190000,-0.530000,-16.440000,92.130000,1.170000,3.000000,True,Green
Best Maximum Sharpe,6.630000,8.620000,0.300000,-18.100000,65.230000,2.040000,4.000000,True,Green
Best Risk Parity,6.510000,8.190000,0.300000,-17.110000,65.450000,2.050000,2.000000,True,Green
Best Minimum ES,0.720000,6.180000,-0.540000,-16.410000,92.670000,1.160000,1.000000,True,Green


In [48]:
final_recommendation = pd.DataFrame(
    {
        "Metric": [
            "Recommended Portfolio",
            "AGG Weight",
            "GLD Weight",
            "XLK Weight",
            "OOS Annualized Return",
            "OOS Volatility",
            "OOS Sharpe Ratio",
            "OOS Maximum Drawdown",
            "Stress Rank",
            "99% Historical VaR Validation",
            "Latest Basel Zone"
        ],

        "Result": [
            "Best Risk Parity",
            f"{risk_parity_weights[0] * 100:.2f}%",
            f"{risk_parity_weights[1] * 100:.2f}%",
            f"{risk_parity_weights[2] * 100:.2f}%",
            f"{institutional_comparison.loc['Best Risk Parity', 'Annualized Return'] * 100:.2f}%",
            f"{institutional_comparison.loc['Best Risk Parity', 'Volatility'] * 100:.2f}%",
            f"{institutional_comparison.loc['Best Risk Parity', 'Sharpe Ratio']:.2f}",
            f"{institutional_comparison.loc['Best Risk Parity', 'Maximum Drawdown'] * 100:.2f}%",
            f"{int(institutional_comparison.loc['Best Risk Parity', 'Overall Stress Rank'])}",
            str(
                institutional_comparison.loc[
                    "Best Risk Parity",
                    "Overall Validation Pass"
                ]
            ),
            institutional_comparison.loc[
                "Best Risk Parity",
                "Latest Basel Zone"
            ]
        ]
    }
)

final_recommendation

,Metric,Result
0,Recommended Portfolio,Best Risk Parity
1,AGG Weight,65.45%
2,GLD Weight,17.79%
3,XLK Weight,16.76%
4,OOS Annualized Return,6.51%
5,OOS Volatility,8.19%
6,OOS Sharpe Ratio,0.30
7,OOS Maximum Drawdown,-17.11%
8,Stress Rank,2
9,99% Historical VaR Validation,True


## Limitations and Monitoring Requirements

The recommended portfolio should not be interpreted as permanently optimal.

Several limitations remain.

1. The optimized weights depend on historical return and covariance estimates.

2. The Risk Parity allocation remains materially exposed to fixed income,
   particularly through its approximately 65% allocation to AGG.

3. Correlations and volatility relationships may change materially during
   future market regimes.

4. Historical VaR validation supports the selected risk model over the
   available out-of-sample period but cannot guarantee future tail-risk
   performance.

5. Hypothetical stress scenarios are deterministic assumptions rather than
   forecasts.

6. Transaction costs, taxes, liquidity requirements and the endowment's
   exact future spending schedule are not fully represented in the current
   optimization.

The portfolio should therefore be monitored periodically using updated
risk contributions, drawdowns, VaR exceptions and stress-test results.

In [49]:
risk_contributions.to_csv(
    TABLE_DIR
    / "portfolio_risk_contributions.csv",
    index=False
)

portfolio_risk_summary.to_csv(
    TABLE_DIR
    / "portfolio_risk_contribution_summary.csv"
)

diversification_summary.to_csv(
    TABLE_DIR
    / "portfolio_diversification_summary.csv"
)

institutional_comparison.to_csv(
    TABLE_DIR
    / "institutional_portfolio_comparison.csv"
)

decision_gates.to_csv(
    TABLE_DIR
    / "institutional_decision_gates.csv"
)

recommended_allocation.to_csv(
    TABLE_DIR
    / "recommended_portfolio_allocation.csv",
    index=False
)

recommended_risk_contributions.to_csv(
    TABLE_DIR
    / "recommended_portfolio_risk_contributions.csv",
    index=False
)

final_evidence_table.to_csv(
    TABLE_DIR
    / "final_portfolio_evidence_table.csv"
)

final_recommendation.to_csv(
    TABLE_DIR
    / "final_portfolio_recommendation.csv",
    index=False
)

print(
    "Notebook 08 outputs saved successfully."
)

Notebook 08 outputs saved successfully.


In [50]:
presentation_table = (
    institutional_comparison[
        [
            "Annualized Return",
            "Volatility",
            "Sharpe Ratio",
            "Maximum Drawdown",
            "Largest Weight",
            "Effective Number of Holdings",
            "Overall Stress Rank",
            "Overall Validation Pass"
        ]
    ]
    .copy()
)

for col in [
    "Annualized Return",
    "Volatility",
    "Maximum Drawdown",
    "Largest Weight"
]:
    presentation_table[col] *= 100

presentation_table = (
    presentation_table
    .rename(
        columns={
            "Annualized Return": "OOS Return (%)",
            "Volatility": "OOS Volatility (%)",
            "Sharpe Ratio": "OOS Sharpe",
            "Maximum Drawdown": "OOS Max Drawdown (%)",
            "Largest Weight": "Largest Weight (%)",
            "Effective Number of Holdings": "Effective Holdings",
            "Overall Stress Rank": "Stress Rank",
            "Overall Validation Pass": "99% Historical VaR Pass"
        }
    )
)

presentation_table.round(2)

,OOS Return (%),OOS Volatility (%),OOS Sharpe,OOS Max Drawdown (%),Largest Weight (%),Effective Holdings,Stress Rank,99% Historical VaR Pass
Portfolio,,,,,,,,
Equal Weight Benchmark,13.760000,13.300000,0.730000,-17.370000,14.290000,7.000000,5.000000,True
Best GMVP,0.790000,6.190000,-0.530000,-16.440000,92.130000,1.170000,3.000000,True
Best Maximum Sharpe,6.630000,8.620000,0.300000,-18.100000,65.230000,2.040000,4.000000,True
Best Risk Parity,6.510000,8.190000,0.300000,-17.110000,65.450000,2.050000,2.000000,True
Best Minimum ES,0.720000,6.180000,-0.540000,-16.410000,92.670000,1.160000,1.000000,True


In [51]:
decision_mapping = {
    "Equal Weight Benchmark":
        "Benchmark",

    "Best GMVP":
        "Not Selected",

    "Best Maximum Sharpe":
        "Alternative",

    "Best Risk Parity":
        "Recommended",

    "Best Minimum ES":
        "Not Selected"
}

final_decision_table = (
    presentation_table.copy()
)

final_decision_table[
    "Decision"
] = (
    final_decision_table.index
    .map(
        decision_mapping
    )
)

final_decision_table

,OOS Return (%),OOS Volatility (%),OOS Sharpe,OOS Max Drawdown (%),Largest Weight (%),Effective Holdings,Stress Rank,99% Historical VaR Pass,Decision
Portfolio,,,,,,,,,
Equal Weight Benchmark,13.759861,13.298224,0.731109,-17.367598,14.285714,7.000000,5.000000,True,Benchmark
Best GMVP,0.787801,6.187266,-0.525208,-16.438651,92.125698,1.169707,3.000000,True,Not Selected
Best Maximum Sharpe,6.633059,8.620796,0.301092,-18.100835,65.228121,2.041204,4.000000,True,Alternative
Best Risk Parity,6.513386,8.193104,0.302203,-17.109475,65.454107,2.048557,2.000000,True,Recommended
Best Minimum ES,0.716707,6.180083,-0.537322,-16.410082,92.668436,1.157248,1.000000,True,Not Selected


## Final Selection Rationale

### Equal Weight Benchmark — Benchmark Only

The Equal Weight portfolio achieved the strongest out-of-sample return and
Sharpe ratio and had the greatest capital diversification.

However, it ranked last in the stress-testing framework and experienced
substantially larger losses during severe equity-market crises.

It is therefore retained as a useful benchmark but is not selected for a
capital-preservation-focused endowment mandate.

### GMVP — Not Selected

GMVP achieved low volatility and strong downside protection.

However, its approximately 92% AGG allocation created excessive
concentration and its out-of-sample Sharpe ratio was negative.

The portfolio is therefore considered too defensive and too concentrated
for the final strategic recommendation.

### Maximum Sharpe — Alternative

The Maximum Sharpe portfolio produced stronger out-of-sample return than
GMVP and Minimum ES and maintained a positive Sharpe ratio.

However, it ranked only fourth in stress resilience and therefore provides
less downside protection than Risk Parity.

It remains a reasonable return-oriented alternative but is not the preferred
portfolio for the stated mandate.

### Minimum ES — Not Selected

Minimum ES achieved the strongest overall stress-testing result and provided
excellent protection during severe equity crises.

However, approximately 93% of its capital was allocated to AGG and its
out-of-sample Sharpe ratio was negative.

The portfolio demonstrates the value of downside-risk optimization but is
too concentrated and return-constrained to be selected as the final
institutional portfolio.

### Risk Parity — Recommended

Risk Parity provides the most balanced combination of downside protection,
risk diversification and realized performance.

It ranks second in stress resilience, passes the preferred 99% Historical
VaR validation framework, generates a positive out-of-sample Sharpe ratio,
and distributes portfolio volatility approximately equally across AGG, GLD
and XLK.

It is therefore selected as the preferred strategic candidate.

In [52]:
capital_vs_risk = (
    recommended_risk_contributions[
        [
            "Asset",
            "Weight (%)",
            "Risk Contribution (%)"
        ]
    ]
    .copy()
)

capital_vs_risk.round(2)

,Asset,Weight (%),Risk Contribution (%)
12,AGG,65.450000,30.880000
13,GLD,17.790000,34.540000
14,XLK,16.760000,34.580000


In [53]:
capital_vs_risk_plot = (
    capital_vs_risk
    .set_index(
        "Asset"
    )
)

capital_vs_risk_plot.plot(
    kind="bar",
    figsize=(9, 6)
)

plt.ylabel(
    "Percent (%)"
)

plt.title(
    "Recommended Portfolio: "
    "Capital Weight versus Risk Contribution"
)

plt.xticks(
    rotation=0
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "recommended_capital_vs_risk_contribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_34172\2041113786.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [54]:
risk_contribution_range = (
    recommended_risk_contributions[
        "Risk Contribution"
    ].max()
    -
    recommended_risk_contributions[
        "Risk Contribution"
    ].min()
)

print(
    "Risk contribution range:",
    f"{risk_contribution_range * 100:.2f}%"
)

Risk contribution range: 3.69%


## Why Is 65% AGG Still Called Risk Parity?

Risk Parity does not require equal capital allocations.

AGG has materially lower volatility than GLD and XLK.

Therefore, a larger capital allocation to AGG is required for it to
contribute a similar amount of portfolio volatility.

In the recommended portfolio:

- AGG represents approximately 65% of capital but about 31% of risk,
- GLD represents approximately 18% of capital but about 35% of risk,
- XLK represents approximately 17% of capital but about 35% of risk.

The portfolio therefore achieves substantially more balanced risk exposure
than its capital weights alone would suggest.

In [55]:
institutional_principles = pd.DataFrame(
    {
        "Institutional Requirement": [
            "Capital Preservation",
            "VaR Model Credibility",
            "Stress Resilience",
            "Risk Diversification",
            "Positive Risk-Adjusted Return",
            "Explainability"
        ],

        "Risk Parity Assessment": [
            "Strong",
            "Pass",
            "Strong",
            "Strong",
            "Positive",
            "High"
        ],

        "Evidence": [
            "8.19% OOS volatility and lower crash exposure than Equal Weight",
            "99% Historical VaR passes Kupiec, independence and conditional coverage",
            "Stress rank 2 of 5",
            "Risk contributions approximately 31% / 35% / 35%",
            "OOS Sharpe approximately 0.30",
            "Simple three-asset allocation with transparent risk decomposition"
        ]
    }
)

institutional_principles

,Institutional Requirement,Risk Parity Assessment,Evidence
0,Capital Preservation,Strong,8.19% OOS volatility and lower crash exposure ...
1,VaR Model Credibility,Pass,"99% Historical VaR passes Kupiec, independence..."
2,Stress Resilience,Strong,Stress rank 2 of 5
3,Risk Diversification,Strong,Risk contributions approximately 31% / 35% / 35%
4,Positive Risk-Adjusted Return,Positive,OOS Sharpe approximately 0.30
5,Explainability,High,Simple three-asset allocation with transparent...


## Return-Adequacy Consideration

The recommended Risk Parity portfolio generated an annualized out-of-sample
return of approximately 6.5%.

Whether this return is sufficient for the endowment's long-term real-spending
objective depends on its actual spending rate, inflation objective,
investment costs and future liability requirements.

Because an explicit long-term required-return target has not been imposed in
the current optimization framework, the recommendation should be interpreted
primarily as a risk-controlled strategic allocation.

If a formal required-return objective is specified by the investment
committee, the portfolio should be re-evaluated against that target.

In [56]:
presentation_table.to_csv(
    TABLE_DIR
    / "final_presentation_portfolio_comparison.csv"
)

final_decision_table.to_csv(
    TABLE_DIR
    / "final_portfolio_selection_decision.csv"
)

capital_vs_risk.to_csv(
    TABLE_DIR
    / "recommended_capital_vs_risk.csv",
    index=False
)

institutional_principles.to_csv(
    TABLE_DIR
    / "institutional_recommendation_evidence.csv",
    index=False
)

print(
    "Final presentation tables saved."
)

Final presentation tables saved.


## Marginal VaR, Component VaR and Expected Shortfall Contributions

Volatility risk contribution shows how each asset contributes to total portfolio
volatility, but the investment brief also requires risk budgeting using each
position's marginal VaR / ES contribution.

For a covariance-based parametric VaR approximation,

\[
VaR_p = z_\alpha \sigma_p
\]

where

\[
\sigma_p = \sqrt{w^\top \Sigma w}
\]

The marginal contribution of asset \(i\) to VaR is

\[
MVaR_i
=
z_\alpha
\frac{(\Sigma w)_i}{\sigma_p}
\]

and the component VaR is

\[
CVaR_i = w_i MVaR_i
\]

The component contributions should approximately sum to total portfolio VaR.

Expected Shortfall contributions are estimated empirically using observations
in which the portfolio loss exceeds the selected VaR threshold.

### Component VaR Modelling Note

For the component VaR decomposition, the portfolio mean return is omitted and
VaR is expressed as a volatility-based homogeneous risk measure:

\[
VaR_p = z_\alpha \sigma_p
\]

This zero-mean approximation is used specifically for risk budgeting and Euler
decomposition, so that the individual component VaR values reconcile exactly
to total portfolio VaR.

It does not replace the full Parametric VaR model evaluated in Notebook 05,
where the portfolio mean return is included.

In [57]:
from scipy.stats import norm

VAR_CONFIDENCE = 0.99

z_alpha = norm.ppf(
    VAR_CONFIDENCE
)

risk_parity_row = (
    shortlist[
        shortlist[
            "Portfolio Label"
        ]
        == "Best Risk Parity"
    ]
    .iloc[0]
)

risk_parity_assets = [
    x.strip()
    for x in risk_parity_row[
        "Subset"
    ].split(",")
]

risk_parity_weights = parse_weights(
    risk_parity_row[
        "Weights"
    ]
)
risk_parity_cov = (
    training_returns[
        risk_parity_assets
    ]
    .cov()
    .values
)

portfolio_variance = (
    risk_parity_weights.T
    @ risk_parity_cov
    @ risk_parity_weights
)

portfolio_volatility = np.sqrt(
    portfolio_variance
)

marginal_var = (
    z_alpha
    * (
        risk_parity_cov
        @ risk_parity_weights
    )
    / portfolio_volatility
)

component_var = (
    risk_parity_weights
    * marginal_var
)

total_parametric_var = (
    z_alpha
    * portfolio_volatility
)

marginal_var_table = pd.DataFrame(
    {
        "Asset":
            risk_parity_assets,

        "Weight":
            risk_parity_weights,

        "Marginal VaR":
            marginal_var,

        "Component VaR":
            component_var,

        "Component VaR Share":
            component_var
            / total_parametric_var
    }
)

marginal_var_table

,Asset,Weight,Marginal VaR,Component VaR,Component VaR Share
0,AGG,0.654541,0.004132,0.002704,0.308823
1,GLD,0.177907,0.017003,0.003025,0.345423
2,XLK,0.167552,0.018071,0.003028,0.345754


In [58]:
print(
    "Total Parametric VaR:",
    total_parametric_var
)

print(
    "Sum of Component VaR:",
    component_var.sum()
)

assert np.isclose(
    component_var.sum(),
    total_parametric_var
)

print(
    "Component VaR successfully reconciles "
    "to total portfolio VaR."
)

Total Parametric VaR: 0.008757122716098086
Sum of Component VaR: 0.008757122716098086
Component VaR successfully reconciles to total portfolio VaR.


In [59]:
marginal_var_display = (
    marginal_var_table.copy()
)

marginal_var_display[
    "Weight"
] *= 100

marginal_var_display[
    "Marginal VaR"
] *= 100

marginal_var_display[
    "Component VaR"
] *= 100

marginal_var_display[
    "Component VaR Share"
] *= 100

marginal_var_display.round(3)

,Asset,Weight,Marginal VaR,Component VaR,Component VaR Share
0,AGG,65.454000,0.413000,0.270000,30.882000
1,GLD,17.791000,1.700000,0.302000,34.542000
2,XLK,16.755000,1.807000,0.303000,34.575000


In [60]:
risk_parity_training_returns = (
    training_returns[
        risk_parity_assets
    ]
    .mul(
        risk_parity_weights,
        axis=1
    )
    .sum(
        axis=1
    )
)

risk_parity_training_returns.head()

2004-11-19   -0.002326
2004-11-22    0.001937
2004-11-23   -0.001891
2004-11-24    0.003190
2004-11-26   -0.001202
dtype: float64

In [61]:
historical_var_99 = (
    -risk_parity_training_returns
    .quantile(
        1 - VAR_CONFIDENCE
    )
)

tail_mask = (
    risk_parity_training_returns
    <= -historical_var_99
)

tail_asset_returns = (
    training_returns
    .loc[
        tail_mask,
        risk_parity_assets
    ]
)

component_es = (
    -tail_asset_returns
    .mean()
    * risk_parity_weights
)

historical_es_99 = (
    component_es.sum()
)

component_es_table = pd.DataFrame(
    {
        "Asset":
            risk_parity_assets,

        "Weight":
            risk_parity_weights,

        "Component ES":
            component_es.values,

        "Component ES Share":
            (
                component_es.values
                / historical_es_99
            )
    }
)

component_es_table

,Asset,Weight,Component ES,Component ES Share
0,AGG,0.654541,0.004762,0.326567
1,GLD,0.177907,0.004554,0.312302
2,XLK,0.167552,0.005266,0.361132


Because all five shortlisted portfolios pass the preferred 99% Historical VaR
validation framework, validation is treated as an eligibility requirement
rather than a ranking criterion in the final portfolio decision.

In [62]:
print(
    "Historical VaR 99%:",
    historical_var_99
)

print(
    "Historical ES 99%:",
    historical_es_99
)

print(
    "Tail observations:",
    tail_mask.sum()
)
assert np.isclose(
    marginal_var_table[
        "Component VaR"
    ].sum(),
    total_parametric_var
)
assert np.isclose(
    component_es_table[
        "Component ES"
    ].sum(),
    historical_es_99
)

assert (
    historical_es_99
    >= historical_var_99
)
print(
    "Tail-risk contribution checks passed."
)
print(
    "Component ES successfully reconciles "
    "to total Historical ES."
)

Historical VaR 99%: 0.009392797773316428
Historical ES 99%: 0.014582923192419023
Tail observations: 44
Tail-risk contribution checks passed.
Component ES successfully reconciles to total Historical ES.


In [63]:
component_es_display = (
    component_es_table.copy()
)

component_es_display[
    "Weight"
] *= 100

component_es_display[
    "Component ES"
] *= 100

component_es_display[
    "Component ES Share"
] *= 100

component_es_display.round(3)

,Asset,Weight,Component ES,Component ES Share
0,AGG,65.454000,0.476000,32.657000
1,GLD,17.791000,0.455000,31.230000
2,XLK,16.755000,0.527000,36.113000


In [64]:
risk_budgeting_summary = (
    marginal_var_table[
        [
            "Asset",
            "Weight",
            "Component VaR Share"
        ]
    ]
    .merge(
        component_es_table[
            [
                "Asset",
                "Component ES Share"
            ]
        ],
        on="Asset",
        how="left"
    )
)

risk_budgeting_summary[
    "Weight (%)"
] = (
    risk_budgeting_summary[
        "Weight"
    ]
    * 100
)

risk_budgeting_summary[
    "Component VaR Share (%)"
] = (
    risk_budgeting_summary[
        "Component VaR Share"
    ]
    * 100
)

risk_budgeting_summary[
    "Component ES Share (%)"
] = (
    risk_budgeting_summary[
        "Component ES Share"
    ]
    * 100
)

risk_budgeting_summary[
    [
        "Asset",
        "Weight (%)",
        "Component VaR Share (%)",
        "Component ES Share (%)"
    ]
].round(2)

,Asset,Weight (%),Component VaR Share (%),Component ES Share (%)
0,AGG,65.450000,30.880000,32.660000
1,GLD,17.790000,34.540000,31.230000
2,XLK,16.760000,34.580000,36.110000


## Risk-Budgeting Interpretation

The recommended Risk Parity portfolio is not assessed using capital weights
alone.

Marginal and component VaR identify how each asset contributes to portfolio
risk under the covariance-based parametric framework, while component Expected
Shortfall measures contribution during realized extreme-loss observations.

This provides an additional institutional risk-budgeting perspective beyond
volatility contribution.

If an asset contributes a substantially larger proportion of VaR or Expected
Shortfall than its capital weight, it may represent a concentration of downside
risk even when the portfolio appears diversified by market value.

These measures are therefore used as monitoring and position-sizing diagnostics
rather than as standalone portfolio-selection rules.

In [65]:
marginal_var_table.to_csv(
    TABLE_DIR
    / "recommended_marginal_component_var.csv",
    index=False
)

component_es_table.to_csv(
    TABLE_DIR
    / "recommended_component_es.csv",
    index=False
)

risk_budgeting_summary.to_csv(
    TABLE_DIR
    / "recommended_tail_risk_budgeting_summary.csv",
    index=False
)

print(
    "Marginal VaR and Component ES outputs saved."
)

Marginal VaR and Component ES outputs saved.


Conclusion
The analysis demonstrates that no candidate portfolio dominates across every performance and risk dimension.

The Equal Weight Benchmark produced the strongest out-of-sample return but the weakest stress resilience.

Minimum ES delivered the strongest downside protection but required extreme fixed-income concentration and produced weak out-of-sample risk-adjusted performance.

GMVP displayed similar defensive characteristics and concentration.

Maximum Sharpe improved return potential but provided weaker crisis resilience.

The Risk Parity portfolio provides the strongest overall institutional compromise.

The recommended allocation is approximately:

65.45% AGG
17.79% GLD
16.76% XLK
Despite the unequal capital allocation, estimated volatility risk contributions are approximately balanced across the three assets.

The portfolio also passes the preferred 99% Historical VaR validation framework and ranks second in the stress-testing analysis.

Accordingly, Risk Parity is recommended as the preferred strategic portfolio for the Serendib University Endowment Fund under the stated capital- preservation mandate.

The recommendation remains conditional on ongoing monitoring, future spending requirements and periodic revalidation as market conditions change.

Final Institutional Recommendation
After integrating portfolio performance, downside risk, out-of-sample VaR validation, stress testing, concentration and risk contribution analysis, the Best Risk Parity portfolio is selected as the preferred strategic candidate for the Serendib University Endowment Fund.

Recommended allocation
AGG: approximately 65.45%
GLD: approximately 17.79%
XLK: approximately 16.76%
The recommendation is not based on maximizing return.

The Equal Weight Benchmark generated substantially higher out-of-sample returns, but it ranked last in the stress-testing analysis and experienced considerably greater losses during severe equity-market crises.

Minimum ES achieved the strongest stress-test result, but approximately 93% of its capital was allocated to AGG and it produced negative out-of-sample risk-adjusted performance.

GMVP demonstrated similar concentration and return limitations.

Maximum Sharpe offered greater return potential but ranked below Risk Parity in stress resilience.

Risk Parity provides the most balanced combination of the required characteristics. It passes the preferred 99% Historical VaR validation, ranks second in stress resilience, generates positive out-of-sample risk-adjusted returns and distributes portfolio volatility approximately equally across AGG, GLD and XLK.

The resulting recommendation is therefore consistent with the client's primary principle that avoiding large losses is more important than maximizing gains.

In [66]:
assert simple_returns.shape == (
    5512,
    7
)

assert (
    training_returns.shape
    == (
        4309,
        7
    )
)

assert (
    training_returns.index.max()
    == pd.Timestamp(
        "2021-12-31"
    )
)

assert len(shortlist) == 5

assert (
    institutional_comparison.shape[0]
    == 5
)

assert (
    institutional_comparison[
        "Overall Validation Pass"
    ]
    .all()
)

assert (
    institutional_comparison.loc[
        "Best Risk Parity",
        "Sharpe Ratio"
    ]
    > 0
)

assert (
    institutional_comparison.loc[
        "Best Risk Parity",
        "Overall Stress Rank"
    ]
    <= 3
)

assert (
    institutional_comparison.loc[
        "Best Risk Parity",
        "Largest Weight"
    ]
    < 0.80
)

assert np.isclose(
    recommended_allocation[
        "Weight"
    ].sum(),
    1.0,
    atol=1e-6
)

assert np.isclose(
    recommended_risk_contributions[
        "Risk Contribution"
    ].sum(),
    1.0,
    atol=1e-6
)

assert np.isclose(
    marginal_var_table[
        "Component VaR"
    ].sum(),
    total_parametric_var,
    atol=1e-10
)

assert np.isclose(
    component_es_table[
        "Component ES"
    ].sum(),
    historical_es_99,
    atol=1e-10
)

assert (
    historical_es_99
    >= historical_var_99
)

assert (
    len(
        recommended_allocation
    )
    == 3
)

assert set(
    recommended_allocation[
        "Asset"
    ]
) == {
    "AGG",
    "GLD",
    "XLK"
}

print(
    "All Notebook 08 final "
    "validation checks passed."
)

print(
    "\nFinal recommended portfolio: "
    "Best Risk Parity"
)

All Notebook 08 final validation checks passed.

Final recommended portfolio: Best Risk Parity
